**Centro Magallanes 120069**

In [ ]:
import copernicusmarine
import time
#120069
lat_centro = -54.1001
lon_centro = -71.2129
buffer = 0.15

min_lon = lon_centro - buffer
max_lon = lon_centro + buffer
min_lat = lat_centro - buffer
max_lat = lat_centro + buffer

directorio_hdd = "coleccion/Historico_120069"

print("Iniciando extracción con Buffer ampliado (0.15)...")

for anio in range(2014, 2026):
    fecha_inicio = f"{anio}-01-01"
    fecha_fin = "2025-05-01" if anio == 2025 else f"{anio}-12-31"
        
    nombre_archivo = f"120069_Fisica_{anio}_Micro.nc"
    
    print(f"\n-> Extrayendo año {anio}...")
    
    try:
        copernicusmarine.subset(
            dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
            variables=["thetao", "so", "uo", "vo", "zos", "mlotst"],  # Añadí mlotst y zos para tener más variables físicas
            start_datetime=fecha_inicio,
            end_datetime=fecha_fin,
            minimum_longitude=min_lon,
            maximum_longitude=max_lon,
            minimum_latitude=min_lat,
            maximum_latitude=max_lat,
            minimum_depth=0.49, 
            maximum_depth=0.51,
            output_directory=directorio_hdd,
            output_filename=nombre_archivo,
            force_download=True
        )
        print(f"[OK] Año {anio} guardado.")
        time.sleep(3) 
        
    except Exception as e:
        print(f"[ERROR] Fallo en el año {anio}: {e}")



In [ ]:
#checar esta estupides por que no funciona

import copernicusmarine

# --- MULTIYEAR (past: e.g. 2020–2022) ---
copernicusmarine.subset(
    dataset_id="cmems_mod_glo_bgc_my_0.25deg_P1D-m",
    variables=["chl", "o2"],
    start_datetime="2020-01-01",
    end_datetime="2022-12-31",
    minimum_longitude=lon_centro-0.2,
    maximum_longitude=lon_centro+0.2,
    minimum_latitude=lat_centro-0.2,
    maximum_latitude=lat_centro+0.2,
    minimum_depth=0,
    maximum_depth=1,
    output_directory="data",
    output_filename="bgc_multiyear.nc"
)

# --- NEAR REAL TIME (recent: 2023–2025) ---
copernicusmarine.subset(
    dataset_id="cmems_mod_glo_bgc_anfc_0.25deg_P1D-m",
    variables=["chl", "o2"],
    start_datetime="2023-01-01",
    end_datetime="2025-05-01",
    minimum_longitude=lon_centro-0.2,
    maximum_longitude=lon_centro+0.2,
    minimum_latitude=lat_centro-0.2,
    maximum_latitude=lat_centro+0.2,
    minimum_depth=0,
    maximum_depth=1,
    output_directory="data",
    output_filename="bgc_nrt.nc"
)

In [ ]:
#checar tambien esta estupides por que no funciona
# Chlorophyll and Dissolved Oxygen from biogeochemistry dataset
print("Extrayendo Clorofila y Oxígeno...")

for anio in range(2020, 2026):
    fecha_inicio = f"{anio}-01-01"
    fecha_fin = "2025-05-01" if anio == 2025 else f"{anio}-12-31"
    
    nombre_archivo = f"120069_BGC_{anio}_Micro_Bioquimica.nc"
    
    print(f"\n-> Extrayendo BGC año {anio}...")
    
    try:
        copernicusmarine.subset(
            dataset_id="cmems_mod_glo_bgc_my_0.25deg_P1D-m",
            variables=["chl", "o2"],  # Chlorophyll and dissolved oxygen
            start_datetime=fecha_inicio,
            end_datetime=fecha_fin,
            minimum_longitude=min_lon,
            maximum_longitude=max_lon,
            minimum_latitude=min_lat,
            maximum_latitude=max_lat,
            minimum_depth=0.49,
            maximum_depth=0.51,
            output_directory=directorio_hdd,
            output_filename=nombre_archivo,
            force_download=True
        )
        print(f"[OK] BGC año {anio} guardado.")
        time.sleep(3)
        
    except Exception as e:
        print(f"[ERROR] Fallo en BGC año {anio}: {e}")

In [ ]:
#Estructuración de archivo consolidado para Machine Learning

import xarray as xr
import pandas as pd
import os

# Configuración de rutas
directorio_hdd = "coleccion/Historico_120069"
lista_dataframes_anuales = []

print("Iniciando procesamiento de matrices espaciales hacia formato tabular (Módulo Físico)...")

for anio in range(2020, 2026):
    archivo_fisica = os.path.join(directorio_hdd, f"120069_Fisica_{anio}_Micro.nc")
    
    # Validación de existencia exclusiva para la matriz física
    if os.path.exists(archivo_fisica):
        print(f"-> Extrayendo tensores del año {anio}...")
        
        ds_fis = xr.open_dataset(archivo_fisica)
        
        # Reducción espacial: colapso de las dimensiones lat/lon/depth mediante promedio
        dim_fis_drop = [d for d in ['latitude', 'longitude', 'depth'] if d in ds_fis.dims]
        df_fis = ds_fis.mean(dim=dim_fis_drop).to_dataframe().reset_index()
        
        # Extracción y filtrado de características físicas
        columnas_fis_necesarias = ['time', 'thetao', 'so', 'uo', 'vo', 'zos', 'mlotst']
        df_fis = df_fis[[c for c in columnas_fis_necesarias if c in df_fis.columns]]
        
        lista_dataframes_anuales.append(df_fis)
        
        ds_fis.close()
        
    else:
        print(f"[ADVERTENCIA] Archivo de datos físicos ausente para el año {anio}. Omitiendo extracción.")

# Consolidación final y formateo de features
if lista_dataframes_anuales:
    df_final = pd.concat(lista_dataframes_anuales, ignore_index=True)
    
    # Asignación de nomenclatura estándar para el pipeline de Machine Learning
    df_final.rename(columns={
        'time': 'Fecha',
        'thetao': 'Temperatura_C',
        'so': 'Salinidad_psu',
        'uo': 'Corriente_E_m_s',
        'vo': 'Corriente_N_m_s',
        'zos': 'Altura_Mar_m',
        'mlotst': 'Capa_Mezcla_m'
    }, inplace=True)
    
    # Ordenamiento cronológico e integridad referencial
    df_final.sort_values('Fecha', inplace=True)
    df_final.dropna(subset=['Fecha'], inplace=True)
    
    # Exportación
    ruta_csv = os.path.join(directorio_hdd, "Consolidado_Ambiental_Centro_120069.csv")
    df_final.to_csv(ruta_csv, index=False, encoding='utf-8')
    
    print(f"\nPipeline ETL completado. Matriz de covariables guardada en: {ruta_csv}")
    print(df_final.head())
else:
    print("\nError fatal: No existen tensores NetCDF en el directorio especificado.")

**Centro 120190, sin utilizar por falta de datos parasitarios extensos**

In [ ]:
import copernicusmarine
import time
#120190
# 1. Ampliamos el Buffer para escapar de la "tierra firme"
lat_centro = -53.2107
lon_centro = -72.8302
buffer = 0.15 # ¡Cambio clave! Ahora buscaremos en un radio más amplio

min_lon = lon_centro - buffer
max_lon = lon_centro + buffer
min_lat = lat_centro - buffer
max_lat = lat_centro + buffer

directorio_hdd = "coleccion/Historico_120190"

print("Iniciando extracción con Buffer ampliado (0.15)...")

for anio in range(2020, 2026):
    fecha_inicio = f"{anio}-01-01"
    fecha_fin = "2025-05-01" if anio == 2025 else f"{anio}-12-31"
        
    nombre_archivo = f"110745_Fisica_{anio}_Micro.nc"
    
    print(f"\n-> Extrayendo año {anio}...")
    
    try:
        copernicusmarine.subset(
            dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
            variables=["thetao", "so", "uo", "vo", "zos", "mlotst"],  # Añadí mlotst y zos para tener más variables físicas
            start_datetime=fecha_inicio,
            end_datetime=fecha_fin,
            minimum_longitude=min_lon,
            maximum_longitude=max_lon,
            minimum_latitude=min_lat,
            maximum_latitude=max_lat,
            minimum_depth=0.49, 
            maximum_depth=0.51,
            output_directory=directorio_hdd,
            output_filename=nombre_archivo,
            force_download=True
        )
        print(f"[OK] Año {anio} guardado.")
        time.sleep(3) 
        
    except Exception as e:
        print(f"[ERROR] Fallo en el año {anio}: {e}")



In [ ]:
#guardar?

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1. Carga de datos
ruta_csv = "coleccion/Historico_110069/Consolidado_Ambiental_Centro_110069.csv"
df = pd.read_csv(ruta_csv)

# 2. Reducción Dimensional
# Como el CSV ya fue aplanado en el ETL, agrupamos directamente por 'Fecha'
df_limpio = df.groupby('Fecha').mean(numeric_only=True).reset_index()

# Parseo estricto del formato temporal
df_limpio['Fecha'] = pd.to_datetime(df_limpio['Fecha'])

# 3. Estructuración del Lienzo (Canvas)
fig, axs = plt.subplots(4, 1, figsize=(12, 16), sharex=True)
fig.suptitle('Dinámica Oceanográfica Superficial 2020 - 2025\nCentro 110069', fontsize=16, fontweight='bold', y=0.92)

# Gráfico 1: Temperatura
# Notar el cambio de 'thetao' a 'Temperatura_C'
axs[0].plot(df_limpio['Fecha'], df_limpio['Temperatura_C'], color='#d62728', linewidth=1.2)
axs[0].set_ylabel('Temperatura (°C)', fontweight='bold')
axs[0].set_title('Temperatura Superficial del Mar', loc='left')
axs[0].grid(True, linestyle='--', alpha=0.6)
axs[0].axhline(y=10.5, color='black', linestyle=':', linewidth=1.5, label='Umbral crítico')
axs[0].legend(loc='upper right')

# Gráfico 2: Salinidad
# Notar el cambio de 'so' a 'Salinidad_psu'
axs[1].plot(df_limpio['Fecha'], df_limpio['Salinidad_psu'], color='#1f77b4', linewidth=1.2)
axs[1].set_ylabel('Salinidad (PSU)', fontweight='bold')
axs[1].set_title('Salinidad Superficial', loc='left')
axs[1].grid(True, linestyle='--', alpha=0.6)

# Gráfico 3: Corriente U
# Notar el cambio de 'uo' a 'Corriente_E_m_s'
axs[2].plot(df_limpio['Fecha'], df_limpio['Corriente_E_m_s'], color='#2ca02c', linewidth=0.8, alpha=0.8)
axs[2].set_ylabel('Velocidad (m/s)', fontweight='bold')
axs[2].set_title('Corriente Este-Oeste', loc='left')
axs[2].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axs[2].grid(True, linestyle='--', alpha=0.6)

# Gráfico 4: Corriente V
# Notar el cambio de 'vo' a 'Corriente_N_m_s'
axs[3].plot(df_limpio['Fecha'], df_limpio['Corriente_N_m_s'], color='#9467bd', linewidth=0.8, alpha=0.8)
axs[3].set_ylabel('Velocidad (m/s)', fontweight='bold')
axs[3].set_title('Corriente Norte-Sur', loc='left')
axs[3].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axs[3].grid(True, linestyle='--', alpha=0.6)

# 4. Formateo de Ejes Temporales Multianuales
# Marcadores principales: Años
axs[3].xaxis.set_major_locator(mdates.YearLocator())
axs[3].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Marcadores secundarios: Trimestres
axs[3].xaxis.set_minor_locator(mdates.MonthLocator(interval=3))
axs[3].xaxis.set_minor_formatter(mdates.DateFormatter('%b'))

plt.xlabel('Eje Temporal', fontweight='bold', fontsize=12)

# Ajuste tipográfico
plt.setp(axs[3].xaxis.get_majorticklabels(), rotation=0, fontweight='bold', fontsize=11)
plt.setp(axs[3].xaxis.get_minorticklabels(), rotation=45, fontsize=9)

plt.tight_layout()
plt.subplots_adjust(top=0.88)
plt.show()

In [ ]:
import copernicusmarine

ds = copernicusmarine.subset(
    dataset_id="cmems_obs-wind_glo_phy_my_l3-metopa-ascat-asc-0.125deg_P1D-i",
    minimum_longitude=-75.0,
    maximum_longitude=-75.0,
    minimum_latitude=-37.0,
    maximum_latitude=-37.0,
    variables=["eastward_wind", "northward_wind"],
    start_datetime="2020-01-01",
    end_datetime="2020-01-10"
)

In [ ]:
import copernicusmarine
copernicusmarine.describe(product_id="WIND_GLO_PHY_L3_MY_012_005")